# Explodierende Gradienten und Gradient Clipping bei einem multivariaten RNN

## Lernziele

Dieses Notebook entwickelt ein vollständiges Regressionsbeispiel mit einer synthetischen Raumklima-Zeitreihe. Ein `SimpleRNN` erhält zu jedem Zeitpunkt **zwei Werte gleichzeitig**: Temperatur und relative Luftfeuchtigkeit. Aus den vergangenen 72 Zeitpunkten sagt es genau einen kontinuierlichen Wert voraus – die Temperatur des unmittelbar folgenden Zeitpunkts.

Dabei verfolgen wir sieben zusammenhängende Lernziele:

1. Ein Zeitschritt darf einen Merkmalsvektor enthalten. **„Sequenziell“ bedeutet, dass die Zeitschritte nacheinander verarbeitet werden. Es bedeutet nicht, dass ein Zeitschritt nur einen einzelnen Wert enthalten darf.**
2. Die Tensorform eines Batches lautet `(Batchgröße, Sequenzlänge, Anzahl Merkmale)`.
3. Beim Aufrollen des RNN über 72 Schritte kann Backpropagation Through Time sehr große Gradienten erzeugen.
4. Große Gradienten können Loss und Gewichtsänderungen destabilisieren.
5. `tf.clip_by_global_norm` begrenzt die Gradienten, die der Optimizer tatsächlich anwendet.
6. Clipping kann den Trainingslauf stabilisieren.
7. Die rohe Gradientennorm wird vor dem Clipping berechnet und kann deshalb weiterhin größer als die Grenze sein.

Alle Zufallsquellen erhalten einen festen Seed. Die folgenden Zellen sind von oben nach unten ausführbar; die Parameter sind fest gewählt und keine Parametersuche ist Bestandteil des Notebooks.

In [ ]:
import os

# TensorFlow-Meldungen werden reduziert; die eigentlichen Berechnungen bleiben unverändert.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Reproduzierbarkeit: NumPy und TensorFlow verwenden denselben festen Ausgangspunkt.
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
tf.config.optimizer.set_experimental_options({"disable_meta_optimizer": True})

ANZAHL_ZEITSCHRITTE = 3_600
SEQUENZLAENGE = 72
BATCHGROESSE = 32
RNN_EINHEITEN = 32
EPOCHEN = 25
LERNRATE = 0.009
MOMENTUM = 0.99
REKURRENTER_GAIN = 1.05
CLIP_NORM = 1.0

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (11, 4.5), "font.size": 11})
np.set_printoptions(precision=3, suppress=True)

print("TensorFlow-Version:", tf.__version__)
print(
    f"Feste Trainingsparameter: Batchgröße={BATCHGROESSE}, Epochen={EPOCHEN}, "
    f"SGD-Lernrate={LERNRATE}, Momentum={MOMENTUM}, Clip-Norm={CLIP_NORM}"
)

## Aufbau der multivariaten Raumklimadaten

Die ursprüngliche Merkmalsmatrix soll 3.600 aufeinanderfolgende Zeitpunkte und zwei Spalten besitzen. Spalte 0 ist die Temperatur in Grad Celsius, Spalte 1 die relative Luftfeuchtigkeit in Prozent. Formal ist ein Zeitpunkt also

\[
x_t = [\mathrm{Temperatur}_t,\ \mathrm{Luftfeuchtigkeit}_t].
\]

Die Temperatur kombiniert einen Tagesrhythmus mit 24 Schritten pro Tag, eine langsamere 14-Tage-Komponente, einen leichten langfristigen Trend und moderates Messrauschen. Die Luftfeuchtigkeit reagiert teilweise gegenläufig auf die Temperatur. Zusätzlich hängt sie von der um einen Schritt verzögerten Temperatur ab, besitzt eine eigene langsamere Periode und eigenes Rauschen. Durch dieses eigenständige Signal ist sie weder identisch mit der Temperatur noch vollständig aus ihr bestimmt. Werte werden auf den plausiblen Bereich von 25 bis 95 Prozent begrenzt.

Die beiden Rohdatenplots zeigen einen Ausschnitt von zehn Tagen. Die unterschiedliche Einheit bleibt sichtbar; standardisiert wird erst nach der chronologischen Aufteilung.

In [ ]:
rng = np.random.default_rng(SEED)
zeit = np.arange(ANZAHL_ZEITSCHRITTE, dtype=np.float32)
tagesphase = 2 * np.pi * zeit / 24
langsame_phase = 2 * np.pi * zeit / (24 * 14)

# Mehrere Zeitskalen, Trend und unabhängiges Rauschen ergeben ein nicht triviales Temperatursignal.
temperatur = (
    21.0
    + 3.2 * np.sin(tagesphase - 0.7)
    + 1.1 * np.sin(langsame_phase + 0.4)
    + 0.0008 * zeit
    + rng.normal(0.0, 0.35, ANZAHL_ZEITSCHRITTE)
)

# Die verzögerte Temperatur trägt zeitliche Information bei; eigenes Rauschen verhindert
# eine vollkommen deterministische Beziehung zwischen beiden Merkmalen.
temperatur_verzoegert = np.r_[temperatur[0], temperatur[:-1]]
luftfeuchtigkeit = (
    58.0
    - 2.4 * (temperatur - temperatur.mean())
    - 0.7 * (temperatur_verzoegert - temperatur.mean())
    + 4.0 * np.sin(tagesphase + 1.0)
    + 2.2 * np.sin(2 * np.pi * zeit / (24 * 9) - 0.5)
    + rng.normal(0.0, 1.8, ANZAHL_ZEITSCHRITTE)
)
luftfeuchtigkeit = np.clip(luftfeuchtigkeit, 25.0, 95.0)

rohdaten = np.column_stack([temperatur, luftfeuchtigkeit]).astype(np.float32)
assert rohdaten.shape == (3600, 2)
assert np.isfinite(rohdaten).all()
assert 25.0 <= rohdaten[:, 1].min() and rohdaten[:, 1].max() <= 95.0

print("Form der ursprünglichen Merkmalsmatrix:", rohdaten.shape)
print("Spalte 0: Temperatur, Spalte 1: relative Luftfeuchtigkeit")
print(f"Korrelation der beiden Merkmale: {np.corrcoef(rohdaten.T)[0, 1]:.3f}")

ausschnitt = 24 * 10
fig, achsen = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
achsen[0].plot(zeit[:ausschnitt], temperatur[:ausschnitt], color="tab:red", label="Temperatur")
achsen[0].set(title="Ausschnitt der ursprünglichen Temperaturreihe", ylabel="Temperatur (°C)")
achsen[0].legend()
achsen[1].plot(zeit[:ausschnitt], luftfeuchtigkeit[:ausschnitt], color="tab:blue", label="Luftfeuchtigkeit")
achsen[1].set(
    title="Ausschnitt der ursprünglichen Luftfeuchtigkeitsreihe",
    xlabel="Zeitschritt (Stunde)",
    ylabel="Relative Luftfeuchtigkeit (%)",
)
achsen[1].legend()
fig.tight_layout()
plt.show()

## Chronologische Aufteilung und Standardisierung ohne Data Leakage

Zeitreihendaten dürfen nicht zufällig in Vergangenheit und Zukunft vermischt werden. Deshalb werden zuerst die ersten 2.520 Rohzeitpunkte dem Training, die folgenden 540 der Validierung und die letzten 540 dem Test zugeordnet. Erst **innerhalb** jedes Abschnitts entstehen überlappende Fenster. So überschreitet kein Fenster eine Abschnittsgrenze.

Mittelwert und Standardabweichung werden ausschließlich aus den Trainingsrohdaten berechnet. Würden Validierungs- oder Testwerte ihre eigenen Skalierungsparameter beitragen, gelangte Information aus späteren Zeitpunkten in die Vorbereitung des Modells. Das wäre Data Leakage. Beide Eingabemerkmale werden getrennt skaliert, weil Grad Celsius und Prozent verschiedene Einheiten und Streuungen besitzen. Auch die Zieltemperatur wird mit Mittelwert und Standardabweichung der Trainings-Temperatur standardisiert. Diese beiden Kennwerte bleiben erhalten, damit Prognosen am Ende wieder als verständliche Grad-Celsius-Werte erscheinen.

Bei einer Abschnittslänge von \(n\) liefert ein 72-Schritt-Fenster genau \(n-72\) Beispiele. Daher erwarten wir 2.448 Trainings- sowie jeweils 468 Validierungs- und Testbeispiele.

## Drei RNN-Dimensionen und die Sequence-to-Vector-Aufgabe

Keras erwartet die Eingabeform

\[
(\text{Batchgröße},\ \text{Sequenzlänge},\ \text{Anzahl Merkmale}).
\]

Ein typischer Batch hat hier die Form `(32, 72, 2)`: 32 Sequenzen, 72 nacheinander verarbeitete Zeitpunkte je Sequenz und an jedem Zeitpunkt zwei gleichzeitig eingelesene Merkmale. Eine einzelne Sequenz `X_train[0]` besitzt damit `(72, 2)`, ein einzelner Zeitpunkt `X_train[0, 0]` die Form `(2,)`.

Die Aufgabe ist eine **Sequence-to-Vector-Regression**. Aus den standardisierten Vektoren \(x_1,\ldots,x_{72}\) wird die Temperatur am direkt folgenden Zeitpunkt vorhergesagt. Das Ziel `y_train[0]` bleibt bewusst zweidimensional organisiert: Jedes Beispiel enthält einen Zielvektor der Form `(1,)`, der vollständige Ziel-Tensor also `(Anzahl Beispiele, 1)`. Die Luftfeuchtigkeit dient ausschließlich als Eingabemerkmal und wird nicht versehentlich zur Zielvariable.

In [ ]:
# Zuerst werden die Rohdaten chronologisch getrennt; kein Fenster darf Grenzen überqueren.
train_roh = rohdaten[:2520]
val_roh = rohdaten[2520:3060]
test_roh = rohdaten[3060:]
assert train_roh.shape == (2520, 2)
assert val_roh.shape == (540, 2)
assert test_roh.shape == (540, 2)

# Nur der Trainingsabschnitt bestimmt Lage und Skala beider Merkmale.
train_mittel = train_roh.mean(axis=0)
train_std = train_roh.std(axis=0)
assert np.all(train_std > 0)

def standardisiere(abschnitt):
    return ((abschnitt - train_mittel) / train_std).astype(np.float32)

def erzeuge_fenster(standardisierte_daten):
    '''Erzeugt 72er-Sequenzen und die jeweils direkt folgende Temperatur als (n, 1).'''
    X = np.stack(
        [standardisierte_daten[i : i + SEQUENZLAENGE]
         for i in range(len(standardisierte_daten) - SEQUENZLAENGE)]
    ).astype(np.float32)
    # Spalte 0 ist Temperatur; der Slice 0:1 bewahrt die explizite Zieldimension.
    y = standardisierte_daten[SEQUENZLAENGE:, 0:1].astype(np.float32)
    return X, y

X_train, y_train = erzeuge_fenster(standardisiere(train_roh))
X_val, y_val = erzeuge_fenster(standardisiere(val_roh))
X_test, y_test = erzeuge_fenster(standardisiere(test_roh))

assert X_train.shape == (2448, 72, 2) and y_train.shape == (2448, 1)
assert X_val.shape == (468, 72, 2) and y_val.shape == (468, 1)
assert X_test.shape == (468, 72, 2) and y_test.shape == (468, 1)
assert X_train[0].shape == (72, 2)
assert X_train[0, 0].shape == (2,)
assert y_train[0].shape == (1,)

print("Rohdatenformen:", train_roh.shape, val_roh.shape, test_roh.shape)
print("X_train.shape:", X_train.shape)
print("y_train.shape:", y_train.shape)
print("X_val.shape / y_val.shape:", X_val.shape, y_val.shape)
print("X_test.shape / y_test.shape:", X_test.shape, y_test.shape)
print("\nX_train[0] =\n", X_train[0])
print("\nX_train[0].shape:", X_train[0].shape)
print("X_train[0, 0]:", X_train[0, 0])
print("X_train[0, 0].shape:", X_train[0, 0].shape)
print("y_train[0]:", y_train[0], "mit Form", y_train[0].shape)

# Für das Beispiel-Fenster werden die beiden Spalten wieder in ihre Originaleinheiten gebracht.
beispiel_original = X_train[0] * train_std + train_mittel
fig, achsen = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
achsen[0].plot(np.arange(1, 73), beispiel_original[:, 0], color="tab:red", label="Temperatur im Fenster")
achsen[0].set(title="Beispiel eines 72-Schritt-Eingabefensters", ylabel="Temperatur (°C)")
achsen[0].legend()
achsen[1].plot(np.arange(1, 73), beispiel_original[:, 1], color="tab:blue", label="Luftfeuchtigkeit im Fenster")
achsen[1].set(xlabel="Position im Eingabefenster", ylabel="Relative Luftfeuchtigkeit (%)")
achsen[1].legend()
fig.tight_layout()
plt.show()

## Zusammenhang zwischen Daten, Modell und Regressionsziel

Die Modell-Inputform lautet `(72, 2)`, weil die Batchdimension von Keras dynamisch ergänzt wird: Jedes einzelne Beispiel enthält 72 Zeitpunkte mit jeweils Temperatur und Luftfeuchtigkeit. Eine `SimpleRNN`-Schicht mit 32 Einheiten und ReLU-Aktivierung verdichtet die komplette Eingabesequenz in einen Hidden-State-Vektor.

Die nachfolgende Dense-Schicht benötigt genau **ein** Neuron, denn gesucht ist ein einzelner kontinuierlicher Temperaturwert. Ihre Aktivierung ist linear; Sigmoid oder Softmax würden den Wertebereich unnötig beschränken und wären für Klassenwahrscheinlichkeiten gedacht. Mean Squared Error (MSE) passt als Trainings-Loss, weil quadratische Abweichungen einer kontinuierlichen Zielgröße minimiert werden. Für die Testinterpretation ergänzen wir MAE und RMSE in Grad Celsius: MAE beschreibt den mittleren absoluten Fehler, RMSE gewichtet große Fehler stärker. `Accuracy` wird nicht verwendet, weil exakte Klassentreffer für kontinuierliche Temperaturen keine sinnvolle Metrik sind.

Die rekurrente Matrix startet orthogonal mit einem vorsichtigen Gain von 1,05. Dieser Wert liegt nur knapp über 1: Er begünstigt eine Verstärkung über viele Schritte, ohne den Lauf bereits im ersten Update numerisch zu zerstören. Der kleine zufällige Start der Ausgabeschicht hält die ersten Rückpropagationssignale zusätzlich kontrolliert.

In [ ]:
def erstelle_modell():
    '''Erzeugt genau die zur Sequence-to-Vector-Regression passende Architektur.'''
    eingabe = tf.keras.Input(shape=(SEQUENZLAENGE, 2), name="raumklima_sequenz")
    hidden = tf.keras.layers.SimpleRNN(
        RNN_EINHEITEN,
        activation="relu",
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED + 1),
        # Ein orthogonaler Gain knapp über 1 macht die Langzeitdynamik bewusst empfindlich.
        recurrent_initializer=tf.keras.initializers.Orthogonal(
            gain=REKURRENTER_GAIN, seed=SEED + 3
        ),
        name="einfaches_rnn",
    )(eingabe)
    ausgabe = tf.keras.layers.Dense(
        1,
        activation="linear",
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.02, seed=SEED + 2),
        name="temperaturprognose",
    )(hidden)
    return tf.keras.Model(eingabe, ausgabe)

# Ein einziges Basismodell liefert die Startgewichte für beide Versuchsvarianten.
basismodell = erstelle_modell()
startgewichte = [gewicht.copy() for gewicht in basismodell.get_weights()]
modell_a = erstelle_modell()
modell_b = erstelle_modell()
modell_a.set_weights(startgewichte)
modell_b.set_weights(startgewichte)

for gewicht_a, gewicht_b, start in zip(modell_a.get_weights(), modell_b.get_weights(), startgewichte):
    assert np.allclose(gewicht_a, gewicht_b)
    assert np.allclose(gewicht_a, start)

assert modell_a.input_shape == (None, 72, 2)
assert modell_a.output_shape == (None, 1)
assert modell_a.layers[1].units == 32
assert modell_a.layers[-1].units == 1
assert tf.keras.activations.serialize(modell_a.layers[-1].activation) == "linear"
assert modell_a(X_train[:2]).shape == (2, 1)

modell_a.summary()
print("\nStartgewichte von Modell A und B sind elementweise identisch: ja")

## Backpropagation Through Time und explodierende Gradienten

Der Hidden State eines einfachen RNN lässt sich vereinfacht schreiben als

\[
h_t = f(W_x x_t + W_h h_{t-1} + b).
\]

Dabei ist \(x_t\) die aktuelle Eingabe, \(h_{t-1}\) der vorherige Hidden State, \(W_x\) die Eingangsgewichtsmatrix, \(W_h\) die rekurrente Gewichtsmatrix und \(b\) der Bias. Im Vorwärtslauf werden die 72 Merkmalsvektoren wirklich nacheinander verarbeitet. Dieselbe rekurrente Matrix \(W_h\) wird an jedem Zeitpunkt wiederverwendet.

Für das Training wird dieses RNN gedanklich über alle 72 Zeitpunkte aufgerollt. Backpropagation Through Time (BPTT) führt den Fehlergradienten durch die aufeinanderfolgenden Hidden States zurück. In der Kettenregel treten daher wiederholt Ableitungen auf, in denen \(W_h\) und die lokale Ableitung der Aktivierungsfunktion enthalten sind. Sind die wirksamen Faktoren über längere Abschnitte größer als 1, kann ihr Produkt und damit die Gradientennorm stark anwachsen. Das ist das **Exploding-Gradient-Problem**. Ein solcher Gradient kann in SGD ein sehr großes Update erzeugen, Gewichte in eine ungünstige Region verschieben und Loss sowie folgende Gradienten weiter destabilisieren.

## Gradient Clipping und fairer Versuchsaufbau

In jedem Batch berechnet `tf.GradientTape` zunächst die echten Gradienten des MSE-Loss bezüglich aller trainierbaren Variablen. Die **rohe globale Norm** fasst sie zusammen:

\[
\lVert g \rVert_2 = \sqrt{\sum_k \lVert g_k \rVert_2^2}.
\]

Für Modell A wird dieser Gradient unverändert an SGD übergeben. Für Modell B wendet `tf.clip_by_global_norm(..., 1.0)` bei Bedarf einen gemeinsamen Skalierungsfaktor an. Dadurch bleibt die Richtung des gesamten Gradientenvektors erhalten, aber die **angewendete** globale Norm liegt höchstens bei 1,0. Wichtig: Clipping greift erst nach der Gradientenberechnung ein. Eine große rohe Norm kann also weiterhin auftreten und wird als solche protokolliert.

Beide Modelle verwenden identische Startgewichte, Daten, Validierungsdaten, Batchgrenzen, Batchreihenfolge, Batchgröße, SGD-Lernrate, Momentum und geplante Epochen. Die Batches werden ohne Shuffling in einer festen Reihenfolge verarbeitet. Pro Batch werden die Normen intern gemessen; dargestellt werden ausschließlich Mittelwert und Maximum je vollständig abgeschlossener Epoche. Falls ein Loss, eine Norm oder ein Gewicht nicht endlich wird, stoppt der betreffende Lauf kontrolliert und wird für die Auswertung auf die Gewichte der letzten vollständig abgeschlossenen Epoche zurückgesetzt.

In [ ]:
# Dieselben festen Batchgrenzen werden in beiden Läufen wiederverwendet; es gibt kein Shuffling.
BATCH_STARTS = np.arange(0, len(X_train), BATCHGROESSE)

def trainiere_modell(modell, clipping_aktiv):
    '''Trainiert ein Modell und aggregiert echte GradientTape-Messwerte pro Epoche.'''
    optimizer = tf.keras.optimizers.SGD(learning_rate=LERNRATE, momentum=MOMENTUM)
    verlauf = {
        "epoche": [], "train_loss": [], "val_loss": [],
        "roh_mittel": [], "roh_max": [],
        "angewendet_mittel": [], "angewendet_max": [],
        "clip_anteil": [],
    }
    letzte_vollstaendige_gewichte = [w.copy() for w in modell.get_weights()]
    nicht_endlich = False
    abbruch_info = "kein Abbruch"

    @tf.function
    def trainingsschritt(x_batch, y_batch):
        # GradientTape differenziert den tatsächlich berechneten Regressions-Loss.
        with tf.GradientTape() as tape:
            prognose = modell(x_batch, training=True)
            loss = tf.reduce_mean(tf.square(y_batch - prognose))
        gradienten = tape.gradient(loss, modell.trainable_variables)
        rohe_norm = tf.linalg.global_norm(gradienten)

        # Nur Modell B begrenzt die globale Norm vor dem Gewichtsupdate.
        if clipping_aktiv:
            angewendete_gradienten, _ = tf.clip_by_global_norm(gradienten, CLIP_NORM)
        else:
            angewendete_gradienten = gradienten
        angewendete_norm = tf.linalg.global_norm(angewendete_gradienten)
        optimizer.apply_gradients(zip(angewendete_gradienten, modell.trainable_variables))
        return loss, rohe_norm, angewendete_norm

    for epoche in range(1, EPOCHEN + 1):
        batch_losses, rohe_normen, angewendete_normen, clip_flags = [], [], [], []
        epoche_vollstaendig = True

        for start in BATCH_STARTS:
            ende = min(start + BATCHGROESSE, len(X_train))
            loss, rohe_norm, angewendete_norm = trainingsschritt(
                X_train[start:ende], y_train[start:ende]
            )
            werte = np.array([loss.numpy(), rohe_norm.numpy(), angewendete_norm.numpy()])
            gewichte_endlich = all(np.isfinite(w).all() for w in modell.get_weights())

            if not np.isfinite(werte).all() or not gewichte_endlich:
                nicht_endlich = True
                epoche_vollstaendig = False
                abbruch_info = f"nicht-endlicher Wert in Epoche {epoche}"
                break

            batch_losses.append(float(loss))
            rohe_normen.append(float(rohe_norm))
            angewendete_normen.append(float(angewendete_norm))
            clip_flags.append(float(clipping_aktiv and rohe_norm > CLIP_NORM))

        if not epoche_vollstaendig:
            print(f"Kontrollierter Abbruch: {abbruch_info}; unvollständige Epoche wird nicht gespeichert.")
            break

        val_prognose = modell(X_val, training=False)
        val_loss = float(tf.reduce_mean(tf.square(y_val - val_prognose)))
        if not np.isfinite(val_loss):
            nicht_endlich = True
            abbruch_info = f"nicht-endlicher Validierungs-Loss in Epoche {epoche}"
            print(f"Kontrollierter Abbruch: {abbruch_info}.")
            break

        # Nur vollständig abgeschlossene Epochen gelangen in die spätere Visualisierung.
        verlauf["epoche"].append(epoche)
        verlauf["train_loss"].append(float(np.mean(batch_losses)))
        verlauf["val_loss"].append(val_loss)
        verlauf["roh_mittel"].append(float(np.mean(rohe_normen)))
        verlauf["roh_max"].append(float(np.max(rohe_normen)))
        verlauf["angewendet_mittel"].append(float(np.mean(angewendete_normen)))
        verlauf["angewendet_max"].append(float(np.max(angewendete_normen)))
        verlauf["clip_anteil"].append(float(np.mean(clip_flags)))
        letzte_vollstaendige_gewichte = [w.copy() for w in modell.get_weights()]

        print(
            f"Epoche {epoche:02d}/{EPOCHEN} | Loss {verlauf['train_loss'][-1]:.4f} | "
            f"Val {val_loss:.4f} | roh max {verlauf['roh_max'][-1]:.3f} | "
            f"angewendet max {verlauf['angewendet_max'][-1]:.3f} | "
            f"geclippt {verlauf['clip_anteil'][-1]:.1%}"
        )

    # Nach einem Abbruch wird ein endlicher, klar dokumentierter Epochenstand ausgewertet.
    if nicht_endlich:
        modell.set_weights(letzte_vollstaendige_gewichte)
    verlauf["nicht_endlich"] = nicht_endlich
    verlauf["abbruch_info"] = abbruch_info
    verlauf["batch_starts"] = BATCH_STARTS.copy()
    return verlauf

def evaluiere_in_grad_celsius(modell):
    '''Transformiert Ziel und Prognose mit den Trainings-Temperaturstatistiken zurück.'''
    prognose_std = modell(X_test, training=False).numpy()
    ziel_c = y_test * train_std[0] + train_mittel[0]
    prognose_c = prognose_std * train_std[0] + train_mittel[0]
    endlich = np.isfinite(prognose_c).all()
    if endlich:
        mae = float(np.mean(np.abs(ziel_c - prognose_c)))
        rmse = float(np.sqrt(np.mean(np.square(ziel_c - prognose_c))))
    else:
        mae, rmse = np.nan, np.nan
    return {"ziel_c": ziel_c, "prognose_c": prognose_c, "mae": mae, "rmse": rmse, "endlich": endlich}

def plot_loss(verlauf, modellname, y_limits):
    plt.figure(figsize=(10, 4.5))
    plt.plot(verlauf["epoche"], verlauf["train_loss"], marker="o", label="Trainings-Loss (MSE)")
    plt.plot(verlauf["epoche"], verlauf["val_loss"], marker="s", label="Validierungs-Loss (MSE)")
    plt.title(f"{modellname}: Trainings- und Validierungs-Loss pro Epoche")
    plt.xlabel("Epoche"); plt.ylabel("MSE (standardisierte Temperatur)")
    plt.ylim(y_limits); plt.legend(); plt.tight_layout(); plt.show()

def plot_gradienten(verlauf, modellname, y_limits, clipping_aktiv):
    fig, achsen = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
    for achse, suffix, titel in zip(achsen, ["mittel", "max"], ["Mittlere Norm", "Maximale Norm"]):
        achse.plot(verlauf["epoche"], verlauf[f"roh_{suffix}"], marker="o", label="Rohe Gradientennorm")
        achse.plot(verlauf["epoche"], verlauf[f"angewendet_{suffix}"], marker="s", linestyle="--", label="Angewendete Gradientennorm")
        if clipping_aktiv:
            achse.axhline(CLIP_NORM, color="black", linestyle=":", label="Clipping-Grenze 1,0")
        achse.set_yscale("log"); achse.set_ylim(y_limits)
        achse.set(title=titel, xlabel="Epoche", ylabel="Globale Gradientennorm")
        achse.legend()
    fig.suptitle(f"{modellname}: rohe und angewendete Gradientennormen pro Epoche")
    fig.tight_layout(); plt.show()

def plot_test(auswertung, modellname, y_limits):
    anzahl = 168
    plt.figure(figsize=(12, 4.5))
    plt.plot(auswertung["ziel_c"][:anzahl, 0], label="Tatsächliche Temperatur", color="black")
    plt.plot(auswertung["prognose_c"][:anzahl, 0], label="Vorhergesagte Temperatur", color="tab:orange")
    plt.title(f"{modellname}: Testvorhersage für die ersten {anzahl} Testbeispiele")
    plt.xlabel("Testbeispiel (chronologisch)"); plt.ylabel("Temperatur (°C)")
    plt.ylim(y_limits); plt.legend(); plt.tight_layout(); plt.show()

## Auswertung Modell A – ohne Gradient Clipping

Modell A erhält die mit `GradientTape` berechneten Gradienten unverändert. Deshalb müssen rohe und angewendete Norm in beiden Gradientendiagrammen exakt übereinanderliegen. Der Loss-Plot zeigt Training und Validierung separat pro vollständig abgeschlossener Epoche; die logarithmische Skala der Gradientenplots macht Änderungen über Größenordnungen sichtbar.

Tritt im folgenden Lauf ein nicht-endlicher Wert auf, wird das ausdrücklich ausgegeben. Die unvollständige Epoche wird weder verschwiegen noch in die Kurven aufgenommen. Für eine noch sinnvolle Testprognose verwendet die Auswertung anschließend den gespeicherten Stand der letzten vollständigen Epoche. Die Temperaturkurven und MAE/RMSE werden aus der standardisierten Darstellung zurück in Grad Celsius transformiert.

In [ ]:
print("Training von Modell A – ohne Gradient Clipping")
verlauf_a = trainiere_modell(modell_a, clipping_aktiv=False)
auswertung_a = evaluiere_in_grad_celsius(modell_a)

# Die mit Modell A festgelegten Achsengrenzen werden später unverändert für Modell B genutzt.
loss_max_a = max(verlauf_a["train_loss"] + verlauf_a["val_loss"])
LOSS_LIMITS = (0.0, 1.08 * loss_max_a)
positive_normen_a = np.array(verlauf_a["roh_mittel"] + verlauf_a["roh_max"])
GRAD_LIMITS = (
    0.1,  # So bleiben auch die kleineren Normen von Modell B auf derselben Skala sichtbar.
    10 ** np.ceil(np.log10(max(positive_normen_a.max(), CLIP_NORM))),
)
erste_testspanne = np.r_[auswertung_a["ziel_c"][:168, 0], auswertung_a["prognose_c"][:168, 0]]
PRED_LIMITS = (float(erste_testspanne.min() - 1.0), float(erste_testspanne.max() + 1.0))

plot_loss(verlauf_a, "Modell A – ohne Gradient Clipping", LOSS_LIMITS)
plot_gradienten(verlauf_a, "Modell A – ohne Gradient Clipping", GRAD_LIMITS, clipping_aktiv=False)
if auswertung_a["endlich"]:
    plot_test(auswertung_a, "Modell A – ohne Gradient Clipping", PRED_LIMITS)
else:
    print("Keine Testkurve: Die Vorhersagen von Modell A sind nicht endlich.")

print(f"Abgeschlossene Epochen: {len(verlauf_a['epoche'])}")
print("Nicht-endliche Werte aufgetreten:", verlauf_a["nicht_endlich"])
print(f"Test-MAE: {auswertung_a['mae']:.3f} °C | Test-RMSE: {auswertung_a['rmse']:.3f} °C")

### Interpretation von Modell A

Der reproduzierbare Lauf beendet neun Epochen vollständig und meldet den kontrollierten Abbruch in der zehnten Epoche. Damit sind insbesondere die ersten beiden Epochen endlich; der Lauf kollabiert nicht beim ersten Update. In den gespeicherten Messwerten steigt die maximale rohe Norm von etwa 6,36 in Epoche 1 auf etwa 52,76 in Epoche 8 und 73,63 in Epoche 9 – später also um mehr als eine Größenordnung gegenüber der ersten Epoche. Weil Modell A nichts begrenzt, sind die angewendeten Normen identisch groß.

Parallel wird der Loss deutlich instabil: Der Validierungs-MSE erreicht in Epoche 9 ungefähr 3,56 statt ungefähr 0,10 in Epoche 1. Das Gradientendiagramm und der Loss-Verlauf belegen gemeinsam, dass große Updates hier mit einer Verschlechterung und schließlich einem nicht-endlichen Trainingszustand einhergehen. Die Testkurve verwendet transparent den letzten vollständigen, noch endlichen Stand; ihre MAE- und RMSE-Werte stehen direkt unter den Plots in Grad Celsius.

## Auswertung Modell B – mit Gradient Clipping

Modell B startet mit denselben Gewichten und verarbeitet exakt dieselben Batchgrenzen in derselben Reihenfolge. Der einzige Unterschied im Trainingsschritt ist `tf.clip_by_global_norm` mit der Grenze 1,0. Das Loss-Diagramm und die Gradientendiagramme verwenden dieselben Achsengrenzen wie bei Modell A, sodass der visuelle Vergleich nicht durch automatisch wechselnde Skalen verzerrt wird.

In den Gradientendiagrammen bleiben rohe und angewendete Norm getrennt sichtbar; eine horizontale Linie kennzeichnet die Clip-Grenze. Ein zusätzliches Balkendiagramm zeigt pro Epoche den Anteil der Batches, deren rohe Norm größer als 1,0 war. Dieser Anteil beschreibt, wie häufig Clipping aktiv eingreifen musste – nicht, wie viele Gradienten künstlich erzeugt wurden. Alle Werte stammen weiterhin unmittelbar aus dem echten Trainingslauf.

In [ ]:
print("Training von Modell B – mit Gradient Clipping")
verlauf_b = trainiere_modell(modell_b, clipping_aktiv=True)
auswertung_b = evaluiere_in_grad_celsius(modell_b)

plot_loss(verlauf_b, "Modell B – mit Gradient Clipping", LOSS_LIMITS)
plot_gradienten(verlauf_b, "Modell B – mit Gradient Clipping", GRAD_LIMITS, clipping_aktiv=True)

plt.figure(figsize=(10, 4.5))
plt.bar(verlauf_b["epoche"], verlauf_b["clip_anteil"], color="tab:purple", label="Anteil geclippter Batches")
plt.title("Modell B – Anteil der Batches mit aktivem Gradient Clipping")
plt.xlabel("Epoche"); plt.ylabel("Anteil geclippter Batches")
plt.ylim(0.0, 1.05); plt.legend(); plt.tight_layout(); plt.show()

if auswertung_b["endlich"]:
    plot_test(auswertung_b, "Modell B – mit Gradient Clipping", PRED_LIMITS)
else:
    print("Keine Testkurve: Die Vorhersagen von Modell B sind nicht endlich.")

print(f"Abgeschlossene Epochen: {len(verlauf_b['epoche'])}")
print("Nicht-endliche Werte aufgetreten:", verlauf_b["nicht_endlich"])
print(f"Test-MAE: {auswertung_b['mae']:.3f} °C | Test-RMSE: {auswertung_b['rmse']:.3f} °C")

### Interpretation von Modell B

Modell B beendet alle 25 vorgesehenen Epochen mit endlichen Werten. Die rohe Maximalnorm überschreitet die Grenze tatsächlich – im ausgeführten Lauf erreicht sie etwa 8,43. Die angewendete Maximalnorm bleibt dagegen in jeder Epoche bei höchstens 1,0. Genau diese Trennung demonstriert, dass Clipping große Rohgradienten nicht verhindert oder nachträglich aus der Messung entfernt; es begrenzt nur den Vektor, den SGD für das Update erhält.

Der Balkenplot zeigt außerdem, dass Clipping nicht pauschal jeden Batch verändert. Es greift nur ein, wenn die rohe globale Norm die Grenze überschreitet. Im Vergleich zur eskalierenden Validierungskurve von Modell A bleibt der Loss von Modell B in einem engen, endlichen Bereich, und die Testkurve bleibt interpretierbar. Die konkreten Fehler in Grad Celsius werden unter den Plots sowie in der folgenden Vergleichstabelle ausgewiesen.

## Direkter Modellvergleich

Erst nachdem beide Varianten separat untersucht wurden, werden ihre wichtigsten Kennzahlen zusammengeführt. Der finale Trainings- und Validierungs-Loss bezieht sich jeweils auf die letzte vollständig gespeicherte Epoche. Die Maximalnormen sind Maxima über alle vollständigen Epochen. Bei Modell A markiert die Spalte „Nicht-endliche Werte“ den dokumentierten Abbruch; seine Testmetriken stammen wegen des Rücksetzens vom letzten endlichen Epochenstand.

Der kleine Vergleichsplot zeigt nur MAE und RMSE in Grad Celsius. Er ersetzt die getrennten Loss-, Gradienten-, Clipping- und Prognoseplots nicht, sondern verdichtet die Testgüte auf zwei geeignete Regressionsmetriken.

In [ ]:
def zeile_fuer_tabelle(name, verlauf, auswertung):
    return {
        "Modell": name,
        "Finaler Trainings-Loss": verlauf["train_loss"][-1],
        "Finaler Validierungs-Loss": verlauf["val_loss"][-1],
        "Test-MAE (°C)": auswertung["mae"],
        "Test-RMSE (°C)": auswertung["rmse"],
        "Max. rohe Gradientennorm": max(verlauf["roh_max"]),
        "Max. angewendete Gradientennorm": max(verlauf["angewendet_max"]),
        "Abgeschlossene Epochen": len(verlauf["epoche"]),
        "Nicht-endliche Werte": "ja" if verlauf["nicht_endlich"] else "nein",
    }

vergleich = pd.DataFrame([
    zeile_fuer_tabelle("A – ohne Clipping", verlauf_a, auswertung_a),
    zeile_fuer_tabelle("B – mit Clipping", verlauf_b, auswertung_b),
]).set_index("Modell")
display(vergleich.round(4))

metriken = vergleich[["Test-MAE (°C)", "Test-RMSE (°C)"]]
ax = metriken.plot(kind="bar", figsize=(9, 4.5), color=["tab:green", "tab:blue"])
ax.set_title("Kompakter Testvergleich der beiden RNN-Varianten")
ax.set_xlabel("Modell"); ax.set_ylabel("Fehler (°C)")
ax.tick_params(axis="x", rotation=0); ax.legend(title="Regressionsmetrik")
plt.tight_layout(); plt.show()

# Fachliche und technische Abschlussprüfungen des vollständig ausgeführten Notebooks.
assert rohdaten.shape == (3600, 2)
assert X_train.shape == (2448, 72, 2) and y_train.shape == (2448, 1)
assert X_val.shape == (468, 72, 2) and X_test.shape == (468, 72, 2)
assert np.array_equal(verlauf_a["batch_starts"], verlauf_b["batch_starts"])
assert len(verlauf_a["epoche"]) >= 2
assert max(verlauf_a["roh_max"][2:]) >= 10 * verlauf_a["roh_max"][0]
assert np.allclose(verlauf_a["roh_max"], verlauf_a["angewendet_max"])
assert len(verlauf_b["epoche"]) == EPOCHEN and not verlauf_b["nicht_endlich"]
assert max(verlauf_b["roh_max"]) > CLIP_NORM
assert max(verlauf_b["angewendet_max"]) <= CLIP_NORM + 1e-5
assert max(verlauf_b["clip_anteil"]) > 0
assert auswertung_a["endlich"] and auswertung_b["endlich"]
assert modell_b.input_shape[1:] == (72, 2) and modell_b.output_shape[-1] == 1

print("Alle Abschlussprüfungen bestanden: Formen, Vergleichbarkeit, echte Epochenaggregate und Clipping-Grenze stimmen.")

### Interpretation des direkten Vergleichs

Die Tabelle bestätigt die unterschiedliche Trainingsdynamik quantitativ. Modell A schließt nur neun der 25 vorgesehenen Epochen ab und verzeichnet nicht-endliche Werte; seine maximale rohe **und angewendete** Gradientennorm liegt bei ungefähr 73,63. Modell B schließt alle 25 Epochen ohne nicht-endliche Werte ab. Zwar erreicht auch seine rohe Norm ungefähr 8,43, doch die angewendete Norm überschreitet 1,0 nicht. Damit ist nicht eine andere Ausgangslage, sondern die Begrenzung vor dem Update der entscheidende experimentelle Unterschied.

Auch die Loss- und Testdarstellungen müssen gemeinsam gelesen werden: Der instabile ungeclippte Lauf verschlechtert seinen Validierungs-Loss stark, während der geclippte Lauf endlich und deutlich kontrollierter bleibt. Die exakten MAE- und RMSE-Werte in der Tabelle bewerten beide zurückgesetzten bzw. finalen Modellstände in derselben Einheit Grad Celsius. Der Vergleich belegt für genau dieses Experiment eine stabilisierende Wirkung; er ist keine Garantie, dass Clipping jedes RNN oder jede Hyperparameterwahl automatisch verbessert.

## Abschließende Kernaussagen

- Ein Zeitschritt kann mehrere Merkmale enthalten. Hier werden Temperatur und Luftfeuchtigkeit zu jedem Zeitpunkt gemeinsam als Vektor der Form `(2,)` eingelesen; danach verarbeitet das RNN den nächsten Zeitpunkt. „Sequenziell“ beschreibt diese zeitliche Reihenfolge, nicht die Anzahl der Merkmale.
- Die Eingabeform lautet Batchgröße × Sequenzlänge × Merkmale. Das konkrete Modell sieht typischerweise `(32, 72, 2)`: 32 Sequenzen mit je 72 Zeitpunkten und zwei Merkmalen. Seine lineare Dense-Schicht gibt genau einen Wert aus – die Temperatur des folgenden Zeitpunkts.
- Die rekurrenten Gewichte werden über alle 72 Zeitpunkte wiederverwendet. Bei BPTT wird durch viele aufeinanderfolgende Zustände differenziert. Wiederholt wirkende Faktoren können sich in der Kettenregel multiplizieren und die Gradientennorm stark vergrößern.
- Genau das zeigt Modell A: Nach zunächst endlichen Epochen wächst seine maximale Norm von etwa 6,36 auf etwa 73,63. Die großen, unverändert angewendeten Gradienten gehen mit instabilem Loss einher; die zehnte Epoche wird kontrolliert abgebrochen.
- Gradient Clipping verändert nicht die bereits berechnete rohe Norm. Modell B besitzt weiterhin Rohgradienten oberhalb von 1,0. `tf.clip_by_global_norm` begrenzt jedoch die globale Norm des für das Update verwendeten Vektors auf höchstens 1,0. Im vorliegenden Lauf beendet Modell B dadurch alle 25 Epochen mit endlichen Werten.
- Die Diagramme enthalten nur Epochenaggregate, obwohl jede Batchnorm aus einem echten `tf.GradientTape`-Schritt stammt. Es wurden weder Normen erfunden noch skaliert oder mit künstlichen Ausreißern versehen.
- Clipping ist ein Schutz gegen zu große Updates, aber kein Universalheilmittel. Es beseitigt weder automatisch verschwindende Gradienten noch die grundsätzlichen Schwierigkeiten eines einfachen RNN beim Lernen sehr langfristiger Abhängigkeiten.